In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW
import pandas as pd

#the same notebook for the other 3 splits
df = pd.read_csv("/kaggle/input/recipe2m-split-datasets/train_split_2.csv") 



In [ ]:
batch_size = 10000  
epochs = 4  

In [ ]:
recipe_tokens = ['<INPUT_START> ', '<INPUT_END>', '<NEXT_INPUT> ', '<TITLE_START> ', ' <TITLE_END>', 
                 '<INGR_START> ', '<NEXT_INGR> ', '<INGR_END>', '<NEXT_INSTR> ', '<INSTR_START> ', 
                 '<INSTR_END>', '<RECIPE_START>', '<RECIPE_END>']

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'additional_special_tokens': recipe_tokens})
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))

model_path = "/kaggle/working/gpt2_epoch1_large_batch39.pth" 
state_dict = torch.load(model_path, map_location=torch.device("cuda"))
model.load_state_dict(state_dict)

device = torch.device("cuda")
model.to(device)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
<ipython-input-4-4dbe9136a98e>:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=T

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/gpt2_epoch1_large_batch39.pth'

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

# Function to split data into large batches
def batchify(data, batch_size):
    return [data[i : i + batch_size] for i in range(0, len(data), batch_size)]

# Custom Dataset
class RecipeDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(self.texts[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        return encodings["input_ids"].squeeze(0), encodings["attention_mask"].squeeze(0)




In [ ]:

large_batches = batchify(df["formatted_recipe"].tolist(), batch_size=batch_size) 

In [ ]:
scaler = GradScaler()
accumulation_steps = 8  

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

In [ ]:
start_batch_idx = 0
start_epoch = 2
for epoch in range(epochs):
    if epoch < start_epoch:
        continue
    print(f"Epoch {epoch + 1}/{epochs} - Training Started")
    
    for large_batch_idx, large_batch in enumerate(large_batches):
        if large_batch_idx < start_batch_idx:
            print(f"already finished batch : {large_batch_idx + 1}")
            continue
        print(f"Processing Large Batch {large_batch_idx + 1}/{len(large_batches)}")

        # Create DataLoader for smaller sub-batches
        dataset = RecipeDataset(large_batch, tokenizer)
        sub_loader = DataLoader(dataset, batch_size=8, shuffle=True)

        optimizer.zero_grad()
        batch_loss = 0.0  

        for step, (input_ids, attention_mask) in enumerate(sub_loader):
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)

            with autocast():  # Mixed precision training
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                loss = outputs.loss / accumulation_steps  # Normalize loss for gradient accumulation

            scaler.scale(loss).backward()
            batch_loss += loss.item()
            
            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(sub_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        print(f"Large Batch {large_batch_idx + 1}: Loss = {loss.item():.4f}")

        scheduler.step(batch_loss)
        
        # Save model after processing each large batch
        save_path = f"gpt2_epoch{epoch}_large_batch{large_batch_idx}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved weights at {save_path}")
        try:
            os.remove(f"gpt2_epoch{epoch}_large_batch{large_batch_idx-2}.pth")
            print(f"Deleted: {large_batch_idx-2}")
        except FileNotFoundError:
            print(f"File not found: {large_batch_idx-2}")
        except Exception as e:
            print(f"Error deleting {large_batch_idx-2}: {e}")
    print(f"Epoch {epoch + 1} Completed!\n")

print("Training Finished!")

In [ ]:
print(">>>>")

In [ ]:
import os

files_to_delete = [
    "/kaggle/working/gpt2_epoch0_large_batch51.pth",
    "/kaggle/working/gpt2_epoch0_large_batch52.pth",
    "/kaggle/working/gpt2_epoch1_large_batch0.pth",
    "/kaggle/working/gpt2_epoch1_large_batch1.pth",
    "/kaggle/working/gpt2_epoch1_large_batch2.pth",
    "/kaggle/working/gpt2_epoch1_large_batch3.pth",
    "/kaggle/working/gpt2_epoch1_large_batch4.pth",
    "/kaggle/working/gpt2_epoch1_large_batch5.pth",
    "/kaggle/working/gpt2_epoch1_large_batch6.pth",
    "/kaggle/working/gpt2_epoch1_large_batch7.pth",
    "/kaggle/working/gpt2_epoch1_large_batch8.pth",
    "/kaggle/working/gpt2_epoch1_large_batch9.pth",
    "/kaggle/working/gpt2_epoch1_large_batch10.pth",
    "/kaggle/working/gpt2_epoch1_large_batch11.pth",
    "/kaggle/working/gpt2_epoch1_large_batch12.pth",
    "/kaggle/working/gpt2_epoch1_large_batch13.pth",
    "/kaggle/working/gpt2_epoch1_large_batch14.pth",
    "/kaggle/working/gpt2_epoch1_large_batch15.pth",
    "/kaggle/working/gpt2_epoch1_large_batch16.pth",
    "/kaggle/working/gpt2_epoch1_large_batch17.pth",
    "/kaggle/working/gpt2_epoch1_large_batch18.pth",
    "/kaggle/working/gpt2_epoch1_large_batch19.pth",
    "/kaggle/working/gpt2_epoch1_large_batch20.pth",
    "/kaggle/working/gpt2_epoch1_large_batch21.pth",
    "/kaggle/working/gpt2_epoch1_large_batch22.pth",
    "/kaggle/working/gpt2_epoch1_large_batch23.pth",
    "/kaggle/working/gpt2_epoch1_large_batch24.pth",
    "/kaggle/working/gpt2_epoch1_large_batch25.pth",
    "/kaggle/working/gpt2_epoch1_large_batch26.pth",
    "/kaggle/working/gpt2_epoch1_large_batch27.pth",
    "/kaggle/working/gpt2_epoch1_large_batch28.pth",
    "/kaggle/working/gpt2_epoch1_large_batch29.pth",
    "/kaggle/working/gpt2_epoch1_large_batch30.pth",
    "/kaggle/working/gpt2_epoch1_large_batch31.pth",
    "/kaggle/working/gpt2_epoch1_large_batch32.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch23.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch24.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch25.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch26.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch27.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch28.pth",
    # "/kaggle/working/gpt2_epoch1_large_batch29.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch44.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch45.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch46.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch37.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch38.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch39.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch40.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch27.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch28.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch17.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch18.pth",
    # "/kaggle/working/gpt2_epoch0_large_batch19.pth"
]

for file in files_to_delete:
    try:
        os.remove(file)
        print(f"Deleted: {file}")
    except FileNotFoundError:
        print(f"File not found: {file}")
    except Exception as e:
        print(f"Error deleting {file}: {e}")
